# AIML-01 Fraud Detection: Day 9-10, Account Graph + Graph Neural Network

**Goal (portal step 4):** analyse the **account graph**, train a **graph neural network (GNN)**, and **blend it with XGBoost**.

**A graph in one minute.** Accounts (or transactions) are *nodes*. Money moving between them is an *edge*. A GNN learns from a node's own features *and* from its neighbours, so it can notice "this account only talks to accounts that were already flagged".

**The honest plan (be ready to explain this in your presentation):**
1. **Part A, PaySim.** We first measure whether the PaySim graph has any structure a GNN could use. Your Day 3-4 work already hinted at the answer: senders appear only once. We check it properly and report the result, whichever way it goes.
2. **Part B, Elliptic Bitcoin dataset.** If PaySim has little graph structure, a GNN cannot prove itself there. So we test the *same idea* on a public dataset built for it: 203,769 Bitcoin transactions linked by payment flows, with illicit/licit labels and a time axis. We compare XGBoost, a GNN, and a blend of both.
3. **A result where the GNN does not win is still a valid result.** We report what we find.

Run cells top to bottom. Part A takes about 3 minutes. Part B takes about 10 minutes on the free CPU runtime (faster with a GPU: Runtime > Change runtime type > T4 GPU).

## 1. Setup

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy kagglehub pyarrow
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_recall_curve

os.makedirs("outputs", exist_ok=True); os.makedirs("figures", exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False, "axes.spines.right": False})
warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)

# Part A. Does the PaySim account graph have structure?

We keep only TRANSFER and CASH_OUT, because that is where all the fraud lives (your Day 1-2 finding). Each transaction is an edge from `nameOrig` (sender) to `nameDest` (receiver).

## 2. Load PaySim (same download as Day 1-2)

In [ ]:
CSV_PATH = os.environ.get("PAYSIM_CSV")   # leave as None to download automatically

if CSV_PATH is None:
    try:
        import kagglehub
        folder = kagglehub.dataset_download("moonknightmarvel/paysim")
        found = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
        print("CSV files found:", found)
        CSV_PATH = found[0]
    except Exception as e:
        print("Automatic download failed:", repr(e))
        print("Upload the PaySim CSV (folder icon on the left), then set CSV_PATH = 'your_file.csv' and re-run this cell.")
print("Using:", CSV_PATH)

cols = ["step", "type", "nameOrig", "nameDest", "amount", "isFraud"]
df = pd.read_csv(CSV_PATH, usecols=cols)
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].reset_index(drop=True)
print(f"{len(df):,} TRANSFER/CASH_OUT transactions | {int(df.isFraud.sum()):,} fraud ({100*df.isFraud.mean():.2f}%)")

## 3. Who sends and who receives?
Two simple counts decide whether a graph model has anything to learn from:
- **Do senders repeat?** If every sender appears once, there is no sender history and no chain of activity.
- **Do receivers repeat?** Receivers that collect from many senders are the "hub" pattern a graph can see.

In [ ]:
orig_n = df["nameOrig"].value_counts()
dest_n = df["nameDest"].value_counts()

single_sender_share = float((orig_n == 1).mean())
tx_from_single = float((df["nameOrig"].map(orig_n) == 1).mean())
single_receiver_share = float((dest_n == 1).mean())
both = len(set(orig_n.index) & set(dest_n.index))

print(f"Sender accounts       : {len(orig_n):>10,}   ({100*single_sender_share:.1f}% send exactly once; {100*tx_from_single:.1f}% of transactions come from such senders)")
print(f"Receiver accounts     : {len(dest_n):>10,}   ({100*single_receiver_share:.1f}% receive exactly once; busiest receiver got {int(dest_n.max()):,})")
print(f"Accounts that are BOTH sender and receiver: {both:,}")

## 4. Connected components (groups of accounts linked by any transaction)
If the graph breaks into thousands of tiny two-account islands, there is nothing for neighbours to say. A real fraud ring would show up as a bigger connected group with an unusually high fraud rate.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

codes, uniques = pd.factorize(pd.concat([df["nameOrig"], df["nameDest"]], ignore_index=True))
n_tx = len(df); src, dst = codes[:n_tx], codes[n_tx:]
A = coo_matrix((np.ones(n_tx, dtype=np.int8), (src, dst)), shape=(len(uniques), len(uniques)))
n_comp, comp_label = connected_components(A, directed=False)
comp_size = np.bincount(comp_label)
tx_comp_size = comp_size[comp_label[src]]

bucket = pd.cut(pd.Series(tx_comp_size), bins=[0, 2, 5, 20, np.inf], labels=["2 accounts", "3-5", "6-20", "21+"])
comp_table = pd.DataFrame({"transactions": bucket.value_counts().sort_index(),
                           "fraud": df["isFraud"].groupby(bucket, observed=False).sum()})
comp_table["fraud_rate_%"] = 100 * comp_table["fraud"] / comp_table["transactions"].clip(lower=1)
print(f"{len(uniques):,} accounts form {n_comp:,} connected groups. Largest group: {int(comp_size.max()):,} accounts.")
print("\nTransactions by size of the group they belong to:")
comp_table

## 5. Do fraudsters leave a trail?
Two direct tests for a "fraud ring":
- **Shared receivers:** do several fraudulent transactions pay the *same* receiver (mule accounts)?
- **Money trail:** does the receiver of a fraudulent TRANSFER later appear as a *sender* (cashing out)? If yes, a graph could follow the money.

In [ ]:
fraud = df[df["isFraud"] == 1]
fr_dest_n = fraud["nameDest"].value_counts()
shared_fraud_receivers = int((fr_dest_n >= 2).sum())

ft_dest = pd.Series(fraud.loc[fraud["type"] == "TRANSFER", "nameDest"].unique())
trail_share = float(ft_dest.isin(orig_n.index).mean()) if len(ft_dest) else float("nan")

print(f"Receivers hit by 2+ fraudulent transactions : {shared_fraud_receivers:,} of {len(fr_dest_n):,} fraud receivers")
print(f"Fraud-TRANSFER receivers that ever appear as a sender: {100*trail_share:.1f}%")

structure_weak = single_sender_share > 0.90 and trail_share < 0.20
print("\nREADING:")
if structure_weak:
    print(" -> WEAK structure. Almost every sender is used once and fraud money rarely reappears as a sender.")
    print("    A graph model has very little to learn from on PaySim. That is a finding, not a failure.")
else:
    print(" -> SOME structure. Repeating accounts exist, so graph features are worth testing on PaySim as well.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
vc = dest_n.value_counts().sort_index()
ax[0].loglog(vc.index, vc.values, "o", ms=4, color="#4C78A8")
ax[0].set_xlabel("Transactions received by one account"); ax[0].set_ylabel("Number of accounts")
ax[0].set_title("Receiver in-degree")
ax[1].bar(comp_table.index.astype(str), comp_table["transactions"], color="#4C78A8")
ax[1].set_yscale("log"); ax[1].set_ylabel("Transactions (log scale)"); ax[1].set_xlabel("Size of connected group")
ax[1].set_title("Group sizes")
fig.tight_layout(); fig.savefig("figures/09_paysim_graph.png", dpi=150); plt.show()

paysim_graph = dict(
    transactions=int(len(df)), sender_accounts=int(len(orig_n)), receiver_accounts=int(len(dest_n)),
    single_sender_share=single_sender_share, single_receiver_share=single_receiver_share,
    connected_groups=int(n_comp), largest_group=int(comp_size.max()),
    shared_fraud_receivers=shared_fraud_receivers, fraud_receivers=int(len(fr_dest_n)),
    fraud_transfer_receiver_reappears_as_sender=trail_share, structure_weak=bool(structure_weak))
del df, A, codes, uniques   # free memory before Part B

# Part B. GNN vs XGBoost on the Elliptic Bitcoin dataset

**The dataset** (Elliptic, a blockchain-analytics company): each node is a Bitcoin transaction, each edge a payment flow from one transaction to another.
- **203,769 nodes, 234,355 edges, 49 time steps** (about two weeks each).
- **Labels:** *illicit* (about 2% of nodes), *licit* (about 21%), and *unknown* (the rest, ignored for scoring).
- **165 features per node:** 93 *local* features (the transaction itself) and 72 *aggregated* features (hand-built summaries of its neighbours).

**What we compare**
- XGBoost on the **local** features only.
- XGBoost on **all** features (local + the hand-built neighbour summaries).
- **GraphSAGE** (a GNN) on local features: it has to *learn* from neighbours on its own.
- GraphSAGE on all features.
- A **blend**: a weighted average of the best XGBoost and the best GNN, with the weight chosen on validation data only.

**Split by time, like Day 5-6:** train on steps 1-27, validate on 28-34, test on 35-49. The test period is the future. All numbers are for the **illicit** class.

## 6. Load the dataset
The first run downloads about 100 MB from the PyTorch Geometric servers.

In [ ]:
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.utils import to_undirected

ROOT = "elliptic_data"
try:
    ds = EllipticBitcoinDataset(root=ROOT)
except Exception as e:
    print("Automatic download failed:", repr(e))
    print("\nOption B: download the 3 CSV files from https://www.kaggle.com/datasets/ellipticco/elliptic-data-set")
    print("(elliptic_txs_features.csv, elliptic_txs_edgelist.csv, elliptic_txs_classes.csv),")
    print("create a folder  elliptic_data/raw  in Colab, drag the 3 files into it, and re-run this cell.")
    raise
data = ds[0]

# the PyG object does not keep the time step, so read it from the raw features file
time_step = pd.read_csv(ds.raw_paths[0], header=None, usecols=[0, 1])[1].to_numpy()
assert len(time_step) == data.num_nodes, "time steps do not line up with nodes"

X = data.x.numpy().astype(np.float32)
y = data.y.numpy()                      # 0 = licit, 1 = illicit, 2 = unknown
labeled = y != 2
print(f"{data.num_nodes:,} nodes | {data.edge_index.shape[1]:,} edges | {X.shape[1]} features | {len(np.unique(time_step))} time steps")
if (data.num_nodes, data.edge_index.shape[1], X.shape[1]) != (203769, 234355, 165):
    print("NOTE: sizes differ from the published 203,769 / 234,355 / 165. Check you loaded the standard Elliptic files.")

ei = data.edge_index.numpy()
cross = float((time_step[ei[0]] != time_step[ei[1]]).mean())
print(f"Edges that connect two different time steps: {100*cross:.2f}%  (0% means the future cannot leak into the past through edges)")

## 7. Time-based split

In [ ]:
LOCAL = list(range(93))                  # the transaction's own features
ALL = list(range(X.shape[1]))            # local + the 72 hand-built neighbour summaries

tr = labeled & (time_step <= 27)
va = labeled & (time_step >= 28) & (time_step <= 34)
te = labeled & (time_step >= 35)
for name, m in [("train", tr), ("valid", va), ("test", te)]:
    print(f"{name:5s} steps {int(time_step[m].min()):>2}-{int(time_step[m].max()):>2} | {int(m.sum()):>6,} labelled | {int((y[m]==1).sum()):>5,} illicit ({100*(y[m]==1).mean():.1f}%)")
print(f"\nRandom guessing would score a PR-AUC of about {(y[te]==1).mean():.3f} on the test set.")

## 8. Helper functions
- **PR-AUC** is the main score (same as the fraud project).
- The **threshold** (the score above which we call something illicit) is chosen on the *validation* set only, by maximising F1. We never tune anything on the test set.
- **F1** balances precision (how many alerts are real) and recall (how much illicit activity we catch).

In [ ]:
def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    return float(thr[int(np.nanargmax(f1[:-1]))])

def prf(y_true, pred):
    tp = int(((pred == 1) & (y_true == 1)).sum()); fp = int(((pred == 1) & (y_true == 0)).sum()); fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1); recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return precision, recall, f1

def evaluate(name, p):
    thr = best_f1_threshold(y[va], p[va])
    precision, recall, f1 = prf(y[te], (p[te] >= thr).astype(int))
    return dict(model=name, valid_pr_auc=average_precision_score(y[va], p[va]),
                test_pr_auc=average_precision_score(y[te], p[te]),
                precision=precision, recall=recall, f1=f1, threshold=thr)

probs = {}   # model name -> illicit probability for every node

## 9. XGBoost baselines
Class weights handle the imbalance, exactly as in Day 5-6. Early stopping uses the validation set.

In [ ]:
def run_xgb(cols, seed=0):
    spw = float((y[tr] == 0).sum() / (y[tr] == 1).sum())
    m = xgb.XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=spw, eval_metric="aucpr", early_stopping_rounds=40,
                          tree_method="hist", random_state=seed, n_jobs=-1)
    m.fit(X[tr][:, cols], y[tr], eval_set=[(X[va][:, cols], y[va])], verbose=False)
    return m.predict_proba(X[:, cols])[:, 1]

probs["XGBoost (local features)"] = run_xgb(LOCAL)
probs["XGBoost (all features)"] = run_xgb(ALL)
for k in list(probs):
    print(f"{k:28s} valid PR-AUC {average_precision_score(y[va], probs[k][va]):.3f} | test PR-AUC {average_precision_score(y[te], probs[k][te]):.3f}")

## 10. GraphSAGE (the graph neural network)
**How it works:** each layer replaces a node's features with a mix of its own features and the *average of its neighbours'* features. Two layers means a node "sees" accounts up to two hops away. We treat edges as two-way so information flows both directions.

**Training details:** full graph in memory, only the *training* nodes contribute to the loss, class-weighted loss for the imbalance, early stopping on validation PR-AUC. Because neural networks start from random numbers, results change a little from run to run, so we train **3 times** (3 random seeds) and average. The spread between seeds tells you how much of a difference is just luck.

In [ ]:
from torch_geometric.nn import SAGEConv

EDGES = to_undirected(data.edge_index).to(DEVICE)

class SAGE(torch.nn.Module):
    def __init__(self, d_in, d_hid=128, p=0.3):
        super().__init__()
        self.c1, self.c2 = SAGEConv(d_in, d_hid), SAGEConv(d_hid, d_hid)
        self.out, self.p = torch.nn.Linear(d_hid, 2), p
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.c1(x, ei)), self.p, self.training)
        x = F.dropout(F.relu(self.c2(x, ei)), self.p, self.training)
        return self.out(x)

def train_sage(cols, seed, max_epochs=200, eval_every=5, patience=8):
    torch.manual_seed(seed); np.random.seed(seed)
    xs = X[:, cols]; mu, sd = xs[tr].mean(0), xs[tr].std(0) + 1e-6
    xt = torch.tensor(np.clip((xs - mu) / sd, -10, 10), dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y, dtype=torch.long, device=DEVICE)
    tr_idx = torch.tensor(np.where(tr)[0], device=DEVICE)
    w = torch.tensor([1.0, float((y[tr] == 0).sum() / (y[tr] == 1).sum())], dtype=torch.float32, device=DEVICE)
    model = SAGE(len(cols)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    best, best_p, bad = -1.0, None, 0
    for ep in range(1, max_epochs + 1):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(xt, EDGES)[tr_idx], yt[tr_idx], weight=w)
        loss.backward(); opt.step()
        if ep % eval_every == 0:
            model.eval()
            with torch.no_grad():
                p = F.softmax(model(xt, EDGES), dim=1)[:, 1].cpu().numpy()
            s = average_precision_score(y[va], p[va])
            if s > best: best, best_p, bad = s, p, 0
            else:
                bad += 1
                if bad >= patience: break
    return best_p, best, ep

SEEDS = [0, 1, 2]
seed_spread = {}
for name, cols in [("GraphSAGE (local features)", LOCAL), ("GraphSAGE (all features)", ALL)]:
    ps, test_scores = [], []
    for seed in SEEDS:
        t0 = time.time()
        p, s, ep = train_sage(cols, seed)
        ps.append(p); test_scores.append(average_precision_score(y[te], p[te]))
        print(f"{name} seed {seed}: valid PR-AUC {s:.3f} | {ep} epochs | {time.time()-t0:.0f}s")
    probs[name] = np.mean(ps, axis=0)                  # average of the 3 runs
    seed_spread[name] = float(np.std(test_scores))
    print(f"  -> test PR-AUC per seed: {[round(v, 3) for v in test_scores]}  (spread +/- {seed_spread[name]:.3f})\n")

## 11. The blend
We take the best XGBoost and the best GNN (judged on **validation**) and mix their scores:
`blend = w x GNN + (1 - w) x XGBoost`.
The weight `w` is picked on the validation set. If `w` comes out as 0 or 1, the blend simply copied one model, and that is worth saying in the write-up.

In [ ]:
def best_of(prefix):
    names = [n for n in probs if n.startswith(prefix)]
    return max(names, key=lambda n: average_precision_score(y[va], probs[n][va]))

bx, bg = best_of("XGBoost"), best_of("GraphSAGE")
ws = np.linspace(0, 1, 11)
val_curve = [average_precision_score(y[va], w * probs[bg][va] + (1 - w) * probs[bx][va]) for w in ws]
w_best = float(ws[int(np.argmax(val_curve))])
blend_name = f"Blend ({w_best:.1f} x GNN + {1-w_best:.1f} x XGB)"
probs[blend_name] = w_best * probs[bg] + (1 - w_best) * probs[bx]
print(f"Blending  GNN = {bg}\n          XGB = {bx}\nBest weight on validation: w = {w_best:.1f}")

## 12. Results on the test period (the future)

In [ ]:
rows = [evaluate(k, p) for k, p in probs.items()]
tbl = pd.DataFrame(rows).set_index("model")
display_cols = ["valid_pr_auc", "test_pr_auc", "precision", "recall", "f1"]
print(f"Random guessing: test PR-AUC about {(y[te]==1).mean():.3f}")
tbl[display_cols]

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for k, p in probs.items():
    pr, rc, _ = precision_recall_curve(y[te], p[te])
    ax.plot(rc, pr, lw=2 if k.startswith("Blend") else 1.3, label=f"{k} ({average_precision_score(y[te], p[te]):.2f})")
ax.axhline((y[te] == 1).mean(), color="grey", ls=":", lw=1)
ax.set_xlabel("Recall (illicit found)"); ax.set_ylabel("Precision (alerts that were real)")
ax.set_title("Precision-recall on the test period"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("figures/10_elliptic_pr_curves.png", dpi=150); plt.show()

## 13. Does it keep working as time passes?
Same models, same thresholds (chosen on validation), scored one time step at a time over the test period. A drop here means the world changed under the model. This is exactly the problem Day 11 (drift monitoring) is about. The original Elliptic paper reports that every model degrades after about step 43, when a major dark market shut down and new illicit behaviour appeared.

In [ ]:
show = [bx, bg, blend_name]
thr = {k: best_f1_threshold(y[va], probs[k][va]) for k in show}
steps = sorted(np.unique(time_step[te]))
fig, ax = plt.subplots(figsize=(8, 4))
per_step = {}
for k in show:
    xs, f1s = [], []
    for s in steps:
        m = te & (time_step == s)
        if (y[m] == 1).sum() == 0: continue
        f1s.append(prf(y[m], (probs[k][m] >= thr[k]).astype(int))[2]); xs.append(s)
    per_step[k] = dict(zip(map(int, xs), map(float, f1s)))
    ax.plot(xs, f1s, marker="o", ms=3, label=k)
ax.set_xlabel("Time step"); ax.set_ylabel("F1 on illicit"); ax.set_title("Performance over the test period"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("figures/11_elliptic_f1_over_time.png", dpi=150); plt.show()

## 14. What to conclude (generated from your numbers)

In [ ]:
best_model = tbl["test_pr_auc"].idxmax()
ref = tbl.loc[bx, "test_pr_auc"]
gnn_row, blend_row = tbl.loc[bg], tbl.loc[blend_name]
noise = seed_spread.get(bg, 0.0)

print(f"Best test PR-AUC: {best_model} ({tbl.loc[best_model, 'test_pr_auc']:.3f})")
print(f"Best XGBoost    : {bx} ({ref:.3f})")
print(f"Best GNN        : {bg} ({gnn_row['test_pr_auc']:.3f}, seed spread +/- {noise:.3f})")
print(f"Blend           : {blend_row['test_pr_auc']:.3f}  (weight on GNN = {w_best:.1f})")
print()
d_gnn = gnn_row["test_pr_auc"] - ref
d_blend = blend_row["test_pr_auc"] - ref
print("READING:")
if abs(d_gnn) <= 2 * noise:
    print(f" -> The GNN and XGBoost differ by {d_gnn:+.3f}, within about two seed-spreads ({2*noise:.3f}). Treat them as equal.")
elif d_gnn > 0:
    print(f" -> The GNN beats the best XGBoost by {d_gnn:+.3f} PR-AUC, more than seed noise.")
else:
    print(f" -> XGBoost beats the GNN by {-d_gnn:.3f} PR-AUC, more than seed noise.")
if w_best in (0.0, 1.0):
    print(f" -> The blend picked w = {w_best:.1f}, so it just copied one model. Mixing did not help on validation.")
elif abs(d_blend) <= 2 * noise:
    print(f" -> The blend changes test PR-AUC by {d_blend:+.3f} versus XGBoost, within seed noise. The blend is not clearly better.")
elif d_blend > 0:
    print(f" -> The blend improves on the best XGBoost by {d_blend:+.3f} PR-AUC, more than seed noise.")
else:
    print(f" -> The blend is worse than the best XGBoost by {-d_blend:.3f} PR-AUC.")
print()
print("Write it exactly like this in your README: the numbers above, the split (train 1-27, validate 28-34, test 35-49),")
print("the seed spread, and what Part A found about PaySim:", "almost no graph structure." if paysim_graph["structure_weak"] else "some repeating accounts.")
print("Do not claim the GNN helps unless the READING says so.")

## 15. How this would plug into the platform
On PaySim there is too little graph structure to justify it. On data with real account networks (like Elliptic) the design would be:
1. A **batch job** (for example hourly) builds the graph from recent transactions and runs the GNN to give every account a **graph risk score**.
2. That score is written to **Feast** as one more feature on the account.
3. The live scoring service reads it with the other features, and **XGBoost** (or a blend) makes the final decision in milliseconds.

The GNN stays out of the live request path because graph computation is slow and the neighbourhood changes more slowly than individual payments.

## 16. Save everything for the repository

In [ ]:
results = dict(
    split=dict(train_steps="1-27", valid_steps="28-34", test_steps="35-49",
               train_n=int(tr.sum()), valid_n=int(va.sum()), test_n=int(te.sum()),
               test_illicit_share=float((y[te] == 1).mean())),
    models=tbl.reset_index().to_dict(orient="records"),
    blend=dict(gnn=bg, xgb=bx, weight_on_gnn=w_best),
    gnn_seed_spread=seed_spread, seeds=SEEDS,
    f1_per_step=per_step, paysim_graph=paysim_graph)
json.dump(results, open("outputs/day9-10_results.json", "w"), indent=2, default=float)
print("Saved outputs/day9-10_results.json")

!zip -qr day9-10_outputs.zip outputs figures
try:
    from google.colab import files
    files.download("day9-10_outputs.zip")
except ImportError:
    print("Not in Colab: day9-10_outputs.zip is in the current folder.")